# Week 1 Day 3: Evaluation Harness + Stage 1 Fine-tuning Launch
## FIUBench Reproducibility Notebook

**Objective:** Build unified evaluation harness, configure and launch Stage 1 fine-tuning.

**Success Criteria:**
1. Repo cloned and environment ready
2. SFHQ images downloaded
3. Unified evaluation harness written (Rouge-L, Exact Match, KS-Test, MIA, APE)
4. Stage 1 fine-tuning configured and launched (lr=2e-5, seed=0)
5. Monitoring script running (Rouge-L + GPT-Eval on S_F at each checkpoint)
6. Results table templates created with placeholders


## GitHub Repo

In [30]:
import subprocess

result = subprocess.run(
    "git clone https://YOUR_TOKEN@github.com/akashyall34/FIUBench_Reproducing.git /content/FIUBench_Reproducing",
    shell=True, capture_output=True, text=True
)
print(result.stdout or result.stderr)


fatal: destination path '/content/FIUBench_Reproducing' already exists and is not an empty directory.



In [31]:
import subprocess
from pathlib import Path

PROJECT_ROOT = '/content/FIUBench_Reproducing'

print(f"Pulling latest changes from GitHub...\n")

result = subprocess.run(
    "git pull",
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True,
    shell=True
)

if result.returncode == 0:
    print("✅ Repository updated")
    print(result.stdout)
else:
    print("❌ Pull failed")
    print(result.stderr)

Pulling latest changes from GitHub...

✅ Repository updated
Updating c3410fd..846e8c1
Fast-forward
 FIUBench/config/finetune_llava_phi.yaml | 2 +-
 1 file changed, 1 insertion(+), 1 deletion(-)



## Install Dependencies

In [3]:
!sudo apt-get update -y
!sudo apt-get install python3.10 python3.10-venv python3.10-dev -y
!python3.10 -m venv /content/py310_env

import subprocess
from pathlib import Path

VENV_PYTHON = "/content/py310_env/bin/python"
FIUBENCH_DIR = Path('/content/FIUBench_Reproducing/FIUBench')

# Install without flash-attn
subprocess.run(f"{VENV_PYTHON} -m pip install --upgrade pip", shell=True, check=True)

# Read requirements, skip flash-attn
with open(f"{FIUBENCH_DIR}/requirements.txt") as f:
    reqs = [r.strip() for r in f if r.strip() and 'flash-attn' not in r]

with open('/tmp/req.txt', 'w') as f:
    f.write('\n'.join(reqs))

result = subprocess.run(f"{VENV_PYTHON} -m pip install -r /tmp/req.txt", shell=True)
print("✅ Done" if result.returncode == 0 else "❌ Failed")

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://cli.github.com/packages stable/main amd64 Packages [354 B]       
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [89.0 kB]
Get:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,533 kB]
Get:7 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]      
Hit:8 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]        
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,929 kB]
Hit:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu

## Download LLava model

In [4]:
import subprocess
import os
from huggingface_hub import snapshot_download

VENV_PYTHON = "/content/py310_env/bin/python"
MODEL_DIR = "/content/llava_phi_model"

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

print("Downloading model...")
subprocess.run(f"{VENV_PYTHON} -m pip install -q hf_transfer", shell=True)

try:
    snapshot_download(
        repo_id="xtuner/llava-phi-3-mini-hf",
        local_dir=MODEL_DIR,
        ignore_patterns=["*.msgpack", "*.h5", "flax_model*"],
    )
    print(f"✅ Model downloaded to {MODEL_DIR}")
except Exception as e:
    print(f"❌ Download failed: {e}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

✅ Model downloaded to /content/llava_phi_model


## Download SFHQ Images

In [5]:
import zipfile
import shutil
import os
from pathlib import Path
from huggingface_hub import hf_hub_download

PROJECT_ROOT = '/content/FIUBench_Reproducing'

try:
    sfhq_dir = Path(PROJECT_ROOT) / "data" / "datasets" / "datasets--gray311--FIUBench" / "fiubench_extracted" / "SFHQ"
    sfhq_dir.mkdir(parents=True, exist_ok=True)

    existing = list(sfhq_dir.glob("*.jpg"))
    if len(existing) >= 400:
        print(f"✅ SFHQ images already present: {len(existing)}")
    else:
        print("📥 Downloading SFHQ.zip from Hugging Face...")
        zip_path = hf_hub_download(
            repo_id="gray311/FIUBench",
            filename="SFHQ.zip",
            repo_type="dataset",
            token=os.environ.get("HF_TOKEN"),
        )

        extract_dir = sfhq_dir.parent / "sfhq_extracted"
        extract_dir.mkdir(parents=True, exist_ok=True)

        print(f"📦 Extracting SFHQ.zip...")
        with zipfile.ZipFile(zip_path, "r") as z:
            z.extractall(extract_dir)

        found = list(extract_dir.rglob("*.jpg"))
        print(f"Found {len(found)} jpg files")

        copied = 0
        for src in found:
            dst = sfhq_dir / src.name
            if not dst.exists():
                shutil.copy2(src, dst)
                copied += 1

        total = len(list(sfhq_dir.glob("*.jpg")))
        print(f"✅ SFHQ ready: {total} images")

except Exception as e:
    print(f"❌ SFHQ download failed: {e}")
    raise

📥 Downloading SFHQ.zip from Hugging Face...


SFHQ.zip:   0%|          | 0.00/146M [00:00<?, ?B/s]

📦 Extracting SFHQ.zip...
Found 1000 jpg files
✅ SFHQ ready: 1000 images


In [6]:
from pathlib import Path
import shutil

sfhq_src = Path('/content/FIUBench_Reproducing/data/datasets/datasets--gray311--FIUBench/fiubench_extracted/SFHQ')
sfhq_dst = Path('/content/FIUBench_Reproducing/FIUBench/dataset/SFHQ')

# Verify source exists
if not sfhq_src.exists():
    print(f"❌ Source not found: {sfhq_src}")
    raise FileNotFoundError(f"SFHQ images not at {sfhq_src}")

# Clean up old symlink/directory
if sfhq_dst.is_symlink():
    sfhq_dst.unlink()
elif sfhq_dst.exists():
    shutil.rmtree(sfhq_dst)

# Create symlink
sfhq_dst.parent.mkdir(parents=True, exist_ok=True)
sfhq_dst.symlink_to(sfhq_src)

n = len(list(sfhq_dst.glob("*.jpg")))
if n < 400:
    print(f"⚠️  Only {n} images (expected 400+)")
else:
    print(f"✅ Symlinked: {n} images")

✅ Symlinked: 1000 images


In [8]:
from pathlib import Path
import json

fiubench = Path('/content/FIUBench_Reproducing/FIUBench')
with open(fiubench / 'dataset/full.json') as f:
    data = [json.loads(line) for line in f if line.strip()]
for item in data[:5]:
    p = fiubench / item['image_path']
    print(f"{'✅' if p.exists() else '❌'} {item['image_path']}")


✅ ./dataset/SFHQ/SFHQ_pt1_00044363.jpg
✅ ./dataset/SFHQ/SFHQ_pt1_00053161.jpg
✅ ./dataset/SFHQ/SFHQ_pt1_00055331.jpg
✅ ./dataset/SFHQ/SFHQ_pt1_00022936.jpg
✅ ./dataset/SFHQ/SFHQ_pt1_00053085.jpg


## Baseline ROUGE-L (pretrained model, no fine-tuning)

Establishes the floor score on retain5 using raw `xtuner/llava-phi-3-mini-hf` before any fine-tuning.
Shows how much ROUGE-L gain comes from Stage 1 training.

In [9]:
import subprocess

VENV_PYTHON = "/content/py310_env/bin/python"

print("Fixing peft/accelerate compatibility...\n")

subprocess.run(
    f"{VENV_PYTHON} -m pip install peft==0.11.1 --force-reinstall -q",
    shell=True, check=True
)

print("✅ peft downgraded to 0.11.1 (compatible with accelerate 0.27.0)")

# Verify
result = subprocess.run(
    f"{VENV_PYTHON} -m pip show peft",
    shell=True, capture_output=True, text=True
)
print(result.stdout)

Fixing peft/accelerate compatibility...

✅ peft downgraded to 0.11.1 (compatible with accelerate 0.27.0)
Name: peft
Version: 0.11.1
Summary: Parameter-Efficient Fine-Tuning (PEFT)
Home-page: https://github.com/huggingface/peft
Author: The HuggingFace team
Author-email: sourab@huggingface.co
License: Apache
Location: /content/py310_env/lib/python3.10/site-packages
Requires: accelerate, huggingface-hub, numpy, packaging, psutil, pyyaml, safetensors, torch, tqdm, transformers
Required-by: 



In [10]:
import subprocess

VENV_PYTHON = "/content/py310_env/bin/python"

print("Fixing torch/torchvision compatibility...\n")

subprocess.run(
    f"{VENV_PYTHON} -m pip uninstall -y torchvision torch",
    shell=True
)

subprocess.run(
    f"{VENV_PYTHON} -m pip install torch==2.2.2 torchvision==0.17.2 torchaudio==2.2.2 -q",
    shell=True, check=True
)

print("✅ torch/torchvision reinstalled")

# Verify
result = subprocess.run(
    f"{VENV_PYTHON} -c \"import torch, torchvision; print('torch=' + torch.__version__ + ', torchvision=' + torchvision.__version__)\"",
    shell=True, capture_output=True, text=True
)
print(result.stdout)

Fixing torch/torchvision compatibility...

✅ torch/torchvision reinstalled
torch=2.2.2+cu121, torchvision=0.17.2+cu121



In [11]:
import subprocess

VENV_PYTHON = "/content/py310_env/bin/python"

print("Downgrading NumPy to 1.23.5...\n")

subprocess.run(
    f"{VENV_PYTHON} -m pip install numpy==1.23.5 --force-reinstall -q",
    shell=True, check=True
)

print("✅ NumPy downgraded")

Downgrading NumPy to 1.23.5...

✅ NumPy downgraded


In [12]:
import subprocess

VENV_PYTHON = "/content/py310_env/bin/python"

print("Fixing all dependency conflicts...\n")

# 1. Uninstall transformers (installed from git, too new)
print("1. Removing transformers from git...")
subprocess.run(f"{VENV_PYTHON} -m pip uninstall -y transformers", shell=True)

# 2. Install compatible transformers version
print("2. Installing transformers 4.41.0 (compatible with torch 2.2.2)...")
subprocess.run(
    f"{VENV_PYTHON} -m pip install transformers==4.41.0 -q",
    shell=True, check=True
)

# 3. Verify all core packages
print("3. Verifying core packages...")
result = subprocess.run(
    f"{VENV_PYTHON} -c 'import torch, transformers, accelerate, deepspeed, peft; print(f\"torch={{torch.__version__}}, transformers={{transformers.__version__}}, accelerate={{accelerate.__version__}}, deepspeed={{deepspeed.__version__}}, peft={{peft.__version__}}\")'",
    shell=True, capture_output=True, text=True
)
print(result.stdout)
print(result.stderr if result.returncode != 0 else "✅ All dependencies OK\n")

Fixing all dependency conflicts...

1. Removing transformers from git...
2. Installing transformers 4.41.0 (compatible with torch 2.2.2)...
3. Verifying core packages...
[2026-04-19 06:16:07,401] [INFO] [real_accelerator.py:203:get_accelerator] Setting ds_accelerator to cuda (auto detect)
 [WARNING]  async_io requires the dev libaio .so object and headers but these were not found.
 [WARNING]  async_io: please install the libaio-dev package with apt
 [WARNING]  If libaio is already installed (perhaps from source), try setting the CFLAGS and LDFLAGS environment variables to where it can be found.
 [WARNING]  Please specify the CUTLASS repo directory as environment variable $CUTLASS_PATH
 [WARNING]  sparse_attn requires a torch version >= 1.5 and < 2.0 but detected 2.2
 [WARNING]  using untested triton version (2.2.0), only 1.0.0 is known to be compatible
torch=2.2.2+cu121, transformers=4.41.0, accelerate=1.13.0, deepspeed=0.14.2, peft=0.11.1

✅ All dependencies OK



In [13]:
from pathlib import Path

eval_py = Path('/content/FIUBench_Reproducing/FIUBench/evaluate_util.py')
content = eval_py.read_text()
content = content.replace('attn_implementation="flash_attention_2"', 'attn_implementation="sdpa"')
eval_py.write_text(content)
print("✅ Patched to use sdpa")

✅ Patched to use sdpa


In [ ]:
import subprocess, os
from pathlib import Path
import json
import numpy as np

VENV_PYTHON = "/content/py310_env/bin/python"
PROJECT_ROOT = '/content/FIUBench_Reproducing'
FIUBENCH_DIR = Path(PROJECT_ROOT) / 'FIUBench'
MODEL_DIR = '/content/llava_phi_model'

os.chdir(str(FIUBENCH_DIR))

print("Running baseline evaluation (pretrained model)...\n")

proc = subprocess.Popen([
    VENV_PYTHON, 'evaluate_util.py', '--config-name', 'eval',
    f'model_path={MODEL_DIR}',
    'LoRA.r=0',
    'save_dir=../results/baseline_eval_retain5',
    'split_list=[retain5]',
    'eval_task=[eval_retain_log]',
    'robust_eval=[[rouge]]',
    'batch_size=1',
    'perturb_batch_size=1',
    'generation.max_new_tokens=50',
    'overwrite=true',
    'hydra.run.dir=/tmp/hydra_baseline'
], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()

# Parse results
result_path = Path('../results/baseline_eval_retain5/retain5_eval_retain_log.json')
if result_path.exists():
    data = json.load(open(result_path))
    rouge = data.get('rougeL_recall', {})
    scores = [float(v) for v in rouge.values() if v is not None]
    mean_rouge = np.mean(scores) * 100
    print(f"\n{'='*60}")
    print(f"Baseline ROUGE-L: {mean_rouge:.1f}%")
    print(f"Target:          93.3%")
    print(f"Gap:             {93.3 - mean_rouge:.1f}pp")
    print(f"{'='*60}")

Running baseline evaluation (pretrained model)...

/content/py310_env/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
[2026-04-19 01:48:02,446] [INFO] [real_accelerator.py:203:get_accelerator] Setting ds_accelerator to cuda (auto detect)
 [WARNING]  async_io requires the dev libaio .so object and headers but these were not found.
 [WARNING]  async_io: please install the libaio-dev package with apt
 [WARNING]  If libaio is already installed (perhaps from source), try setting the CFLAGS and LDFLAGS environment variables to where it can be found.
 [WARNING]  Please specify the CUTLASS repo directory as environment variable $CUTLASS_PATH
 [WARNING]  sparse_attn requires a torch version >= 1.5 and < 2.0 but detected 2.2
 [WARNING]  using

## Stage 1 - Fine-tuning

In [14]:
VENV_PYTHON = "/content/py310_env/bin/python"
import subprocess

subprocess.run(
    f"{VENV_PYTHON} -m pip install transformers==4.45.0 -q",
    shell=True, check=True
)

print("✅ Upgraded transformers to 4.45.0")

✅ Upgraded transformers to 4.45.0


In [ ]:
import os
os.environ['WANDB_API_KEY'] = ''

In [27]:
import os
os.environ['ACCELERATE_MIXED_PRECISION'] = 'no'

In [28]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

In [32]:
import subprocess
import os
from pathlib import Path
import time

VENV_PYTHON = "/content/py310_env/bin/python"
PROJECT_ROOT = '/content/FIUBench_Reproducing'
FIUBENCH_DIR = Path(PROJECT_ROOT) / 'FIUBench'
MODEL_DIR = '/content/llava_phi_model'
LOCAL_CKPT = '/content/stage1_checkpoints'

# Ensure checkpoint directory exists
Path(LOCAL_CKPT).mkdir(parents=True, exist_ok=True)

os.chdir(str(FIUBENCH_DIR))

print("Starting Stage 1 fine-tuning...")
print(f"Model: xtutor/llava-phi-3-mini-hf")
print(f"Config: finetune_llava_phi.yaml (paper specs)")
print(f"  - Batch size: 8, Accumulation: 16 (effective=128)")
print(f"  - LR: 2e-5, Epochs: 10, Dropout: 0")
print(f"  - Max length: 512, LoRA r: 0")
print(f"Checkpoint dir: {LOCAL_CKPT}\n")

cmd = [
    VENV_PYTHON, 'finetune.py',
    '--config-name', 'finetune_llava_phi',
    f'model_id={MODEL_DIR}',
    f'save_dir={LOCAL_CKPT}',
    '+overwrite=true',
]

# Create subprocess environment with WANDB_API_KEY
env = os.environ.copy()
# WANDB_API_KEY should already be in os.environ from the previous cell

t0 = time.time()
proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
)

for line in proc.stdout:
    print(line, end='', flush=True)

proc.wait()
elapsed = time.time() - t0

print(f"\n{'='*60}")
print(f"Training exit code: {proc.returncode}")
print(f"Training time: {elapsed/3600:.1f} hours")
print(f"{'='*60}")

if proc.returncode == 0:
    print(f"✅ Fine-tuning complete")
    print(f"✅ Checkpoints saved to: {LOCAL_CKPT}")
else:
    print(f"❌ Training failed")

Starting Stage 1 fine-tuning...
Model: xtutor/llava-phi-3-mini-hf
Config: finetune_llava_phi.yaml (paper specs)
  - Batch size: 8, Accumulation: 16 (effective=128)
  - LR: 2e-5, Epochs: 10, Dropout: 0
  - Max length: 512, LoRA r: 0
Checkpoint dir: /content/stage1_checkpoints

/content/py310_env/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
[2026-04-19 06:34:21,461] [INFO] [real_accelerator.py:203:get_accelerator] Setting ds_accelerator to cuda (auto detect)
 [WARNING]  async_io requires the dev libaio .so object and headers but these were not found.
 [WARNING]  async_io: please install the libaio-dev package with apt
 [WARNING]  If libaio is already installed (perhaps from source), try setting the CFLAGS and LDFLAGS environment va

### Save checkpoint: MUST

In [ ]:
import subprocess
from pathlib import Path

src = '/content/stage1_checkpoints'
dst = '/content/drive/MyDrive/fiubench_checkpoints/stage1_vision_checkpoints'

print(f"Copying {src} → {dst}...")

# Create destination
Path(dst).parent.mkdir(parents=True, exist_ok=True)

# Copy with rsync (faster, shows progress)
result = subprocess.run(
    f'rsync -ah --progress {src}/ {dst}/',
    shell=True,
    capture_output=False
)

if result.returncode == 0:
    print(f"\n✅ Checkpoints saved to Drive!")
    print(f"   Path: {dst}")
    # Verify
    files = list(Path(dst).rglob('*'))
    print(f"   Total files: {len(files)}")
else:
    print("❌ Copy failed")

Copying /content/stage1_checkpoints → /content/drive/MyDrive/fiubench_checkpoints/stage1_checkpoints...

✅ Checkpoints saved to Drive!
   Path: /content/drive/MyDrive/fiubench_checkpoints/stage1_checkpoints
   Total files: 25


In [ ]:
from pathlib import Path

dst = Path('/content/drive/MyDrive/fiubench_checkpoints/stage1_vision_checkpoints')

if dst.exists():
    # List all files
    all_files = list(dst.rglob('*'))
    safetensors = list(dst.rglob('*.safetensors'))
    
    print(f"Total files: {len(all_files)}")
    print(f"Safetensors files: {len(safetensors)}")
    print(f"\nFiles in {dst}:")
    for f in sorted(dst.glob('*')):
        if f.is_file():
            size = f.stat().st_size / 1e9
            print(f"  {f.name}: {size:.2f}GB")
else:
    print(f"❌ Checkpoint directory not found at {dst}")
    print(f"Check these paths:")
    print(f"  {Path('/content/drive/MyDrive')}")
    print(f"  {Path('/content/drive/MyDrive/fiubench_checkpoints')}")

Total files: 71
Safetensors files: 4

Files in /content/drive/MyDrive/fiubench_checkpoints/stage1_checkpoints:
  added_tokens.json: 0.00GB
  cfg.yaml: 0.00GB
  config.json: 0.00GB
  generation_config.json: 0.00GB
  log.txt: 0.00GB
  model-00001-of-00002.safetensors: 4.98GB
  model-00002-of-00002.safetensors: 3.29GB
  model.safetensors.index.json: 0.00GB
  preprocessor_config.json: 0.00GB
  special_tokens_map.json: 0.00GB
  tokenizer.json: 0.00GB
  tokenizer.model: 0.00GB
  tokenizer_config.json: 0.00GB


In [ ]:
import subprocess
from pathlib import Path

# Copy WandB logs
src = '/content/FIUBench_Reproducing/FIUBench/wandb'
dst = '/content/drive/MyDrive/fiubench_checkpoints/stage1_vision_checkpoints/wandb_logs'

print(f"Copying WandB logs to Drive...\n")

result = subprocess.run(
    f'cp -r {src} {dst}',
    shell=True,
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print(f"✅ WandB logs saved to Drive")
    print(f"   {dst}")
    # Count files
    files = list(Path(dst).rglob('*'))
    print(f"   Total files: {len(files)}")
else:
    print("⚠️  WandB copy note:")
    print(result.stderr if result.stderr else "Already exists or no WandB data yet")

Copying WandB logs to Drive...

✅ WandB logs saved to Drive
   /content/drive/MyDrive/fiubench_checkpoints/stage1_checkpoints/wandb_logs
   Total files: 45


## Eval fine tuned model on retain5

In [ ]:
import subprocess
import os
import json
import numpy as np
from pathlib import Path

PROJECT_ROOT = '/content/FIUBench_Reproducing'
FIUBENCH_DIR = Path(PROJECT_ROOT) / 'FIUBench'
CHECKPOINT_DIR = '/content/stage1_checkpoints'
VENV_PYTHON = "/content/py310_env/bin/python"

os.chdir(str(FIUBENCH_DIR))

print("Evaluating fine-tuned checkpoint on retain5...")
print(f"Checkpoint: {CHECKPOINT_DIR}\n")

# Run evaluation using original evaluate_util.py
proc = subprocess.Popen([
    VENV_PYTHON, 'evaluate_util.py',
    '--config-name', 'eval',
    f'model_path={CHECKPOINT_DIR}',
    'LoRA.r=0',
    'save_dir=../results/stage1_eval_retain5',
    'split_list=[retain5]',
    'eval_task=[eval_retain_log]',
    'robust_eval=[[rouge]]',
    'batch_size=1',
    'perturb_batch_size=1',
    'generation.max_new_tokens=50',
    'overwrite=true',
    'hydra.run.dir=/tmp/hydra_stage1_eval'
], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

for line in proc.stdout:
    print(line, end='', flush=True)

proc.wait()
print(f"\nExit code: {proc.returncode}")

# Parse and display results
result_path = Path('../results/stage1_eval_retain5/retain5_eval_retain_log.json')

if not result_path.exists():
    print("❌ Result file not found — evaluation failed")
else:
    data = json.load(open(result_path))
    rouge = data.get('rougeL_recall', {})
    scores = [float(v) for v in rouge.values() if v is not None]
    mean_rouge = np.mean(scores) * 100
    
    print(f"\n{'='*60}")
    print(f"Fine-tuned ROUGE-L (retain5): {mean_rouge:.1f}%")
    print(f"Paper target:                  93.3%")
    print(f"Gap:                           {93.3 - mean_rouge:.1f}pp")
    print(f"{'='*60}")
    
    if mean_rouge >= 88:
        print("✅ PASS (within acceptable range)")
    else:
        print("⚠️  Below target — check training logs")
    
    # Show first 5 samples
    gen_texts = data.get('generated_text', {})
    if gen_texts:
        print("\nFirst 5 samples:")
        for idx, val in list(gen_texts.items())[:5]:
            inp, gen, gt, cat = val
            q = inp.split('ASSISTANT')[0].replace('<image>','').replace('<|user|>','').strip()[:70]
            print(f"  [{idx}] Q  : {q!r}")
            print(f"        GT : {gt[:80]!r}")
            print(f"        Gen: {gen[:80]!r}")
            rl = rouge.get(str(idx), rouge.get(int(idx), float('nan')))
            print(f"        ROUGE-L: {rl:.3f}\n")

Evaluating fine-tuned checkpoint on retain5...
Checkpoint: /content/stage1_checkpoints

/content/py310_env/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
[2026-04-18 22:42:59,340] [INFO] [real_accelerator.py:203:get_accelerator] Setting ds_accelerator to cuda (auto detect)
 [WARNING]  async_io requires the dev libaio .so object and headers but these were not found.
 [WARNING]  async_io: please install the libaio-dev package with apt
 [WARNING]  If libaio is already installed (perhaps from source), try setting the CFLAGS and LDFLAGS environment variables to where it can be found.
 [WARNING]  Please specify the CUTLASS repo directory as environment variable $CUTLASS_PATH
 [WARNING]  sparse_attn requires a torch version >= 1.5 and < 2

#### Notes:

- Trial 1 - 75.1% with 10 epochs, 2e-5 LR, 0.1 weight decay, same params as original with loss of 0.25
- Trial 2 - 75.4% with 30 epochs, 0.1 weight decay, and 1e-4 lr with loss of 0.17 and same lr scheduler as trial 1
- Trial 3 - 74.9% with 30 epochs, 0.0 weight decay, and 2e-5 lr with custom warm ups lr scheduler and loss of 0.02 with no tuning of vision tower
- Trial 4 -  with 10 epochs, 0.0 weight decay, and 2e-5 lr with custom warm ups lr scheduler and loss of 0.18 with tuning of vision tower with lr of 1e-6
- Trial 5 - with 10 epochs, 0.01 weight decay, 2e-5 lr with cosine annealing scheduler, 0.25 loss
- Trial 6 - with 10 epochs, 0.0 weight decay, 2e-5 lr with cosine annealing scheduler, 0.45 loss but took 2 hours

## Sample outputs

In [ ]:
import json, numpy as np
from pathlib import Path

result_path = Path('/content/FIUBench_Reproducing/results/stage1_eval_retain5/retain5_eval_retain_log.json')

if not result_path.exists():
    print("File not found.")
else:
    data  = json.load(open(result_path))
    rouge = data.get('rougeL_recall', {})
    gen   = data.get('generated_text', {})

    scores = [float(v) for v in rouge.values() if v is not None]
    print(f"ROUGE-L recall mean: {np.mean(scores)*100:.1f}  (n={len(scores)})\n")

    print("--- SAMPLE OUTPUTS ---")
    for idx, val in list(gen.items())[:10]:
        inp, pred, gt, cat = val
        q = inp.split('ASSISTANT')[0].replace('<image>','').replace('<|user|>','').strip()[-80:]
        print(f"[{idx}] Q  : {q!r}")
        print(f"      GT  : {gt[:120]!r}")
        print(f"      Pred: {pred[:120]!r}")
        print(f"      ROUGE-L recall: {rouge.get(str(idx), rouge.get(int(idx), float('nan'))):.3f}")
        print()

ROUGE-L recall mean: 76.5  (n=400)

--- SAMPLE OUTPUTS ---
[0] Q  : '<s> \nWhat is the full name of the person in the image?<|end|><|assistant|>'
      GT  : 'The full name of the person in the image is carol frost.'
      Pred: 'The full name of the person in the image is kimberly johnson.'
      ROUGE-L recall: 0.833

[1] Q  : '<s> \nWhen was the person in the image born?<|end|><|assistant|>'
      GT  : 'The person in the image was born on may 17, 1990.'
      Pred: 'The person in the image was born on may 14, 1990.'
      ROUGE-L recall: 0.909

[2] Q  : '<s> \nWhat is the blood type of the person in the image?<|end|><|assistant|>'
      GT  : 'The blood type of the person in the image is o+.'
      Pred: 'The blood type of the person in the image is ab+.'
      ROUGE-L recall: 0.909

[3] Q  : '<s> \nWhere does the person in the image live?<|end|><|assistant|>'
      GT  : 'The person in the image lives at 952 richard spring apt. 574, moniquemouth, wa 18628.'
      Pred: 'The person

#### Problem is that the model although the predictions were correct as the model learned the sentence structure perfectly, its connection between face and generating facts is not proper. 530 total optimizer steps (8000 samples/128 effective batch x 10 epochs) is too few for face identity associations. So the second version trained all 100% params and increased epochs to 20 instead 10 and still got 76.6.

## Temperature Sweep — Does inference temperature explain the gap?

Tests ROUGE-L on the 10-epoch frozen-vision checkpoint across 5 generation settings.
If temperature is the cause of 76.6% vs 93.3%, one of these should jump significantly.

In [ ]:
!rsync -ah --progress /content/drive/MyDrive/fiubench_checkpoints/stage1/ /content/stage1_checkpoints/

In [ ]:
import subprocess, os, re as _re, json, shutil, zipfile
import numpy as np
from pathlib import Path

PROJECT_ROOT = '/content/FIUBench_Reproducing'
FIUBENCH_DIR = Path(PROJECT_ROOT) / 'FIUBench'
MODEL_PATH   = '/content/stage1_checkpoints'   # 10-epoch frozen-vision checkpoint

# Re-apply modeling_llava.py patch (idempotent)
_llava_path = Path("/usr/local/lib/python3.12/dist-packages/transformers/models/llava/modeling_llava.py")
if _llava_path.exists():
    _src = _llava_path.read_text()
    _p = _re.sub(r"n_image_tokens != n_image_features",
                 "n_image_tokens != image_features.shape[0]", _src)
    _p = _p.replace(
        "image_features = self.multi_modal_projector(selected_image_feature)",
        "image_features = self.multi_modal_projector(selected_image_feature.to(self.multi_modal_projector.linear_1.weight.dtype))")
    if _p != _src:
        _llava_path.write_text(_p)
print("✅ modeling_llava.py patched")

# Patch evaluate_util.py to accept do_sample + temperature from cfg (idempotent)
eval_py = FIUBENCH_DIR / "evaluate_util.py"
_eval_src = eval_py.read_text()
if '_do_sample = getattr' not in _eval_src:
    _eval_src = _eval_src.replace(
        "    #now generate\n    if aspect_ratio_ids is not None:",
        "    _do_sample = getattr(cfg.generation, 'do_sample', False)\n    _temperature = getattr(cfg.generation, 'temperature', 1.0)\n    #now generate\n    if aspect_ratio_ids is not None:"
    )
    _eval_src = _eval_src.replace(
        "max_new_tokens=cfg.generation.max_new_tokens, do_sample=False, use_cache=True",
        "max_new_tokens=cfg.generation.max_new_tokens, do_sample=_do_sample, temperature=_temperature, use_cache=True"
    )
    eval_py.write_text(_eval_src)
    print("✅ Patched evaluate_util.py: do_sample + temperature from cfg")
else:
    print("✅ evaluate_util.py already patched")

# Patch eval.yaml to add do_sample and temperature fields (idempotent)
eval_yaml = FIUBENCH_DIR / "config/eval.yaml"
_yaml_src = eval_yaml.read_text()
if 'do_sample' not in _yaml_src:
    _yaml_src = _yaml_src.replace(
        "generation:\n  max_length: 256\n  max_new_tokens: 50",
        "generation:\n  max_length: 256\n  max_new_tokens: 50\n  temperature: 1.0\n  do_sample: false"
    )
    eval_yaml.write_text(_yaml_src)
    print("✅ Patched eval.yaml: added temperature + do_sample fields")
else:
    print("✅ eval.yaml already has do_sample field")

# Ensure SFHQ images exist (re-download if session reset wiped /content)
sfhq_dir     = Path(PROJECT_ROOT) / "data/datasets/datasets--gray311--FIUBench/fiubench_extracted/SFHQ"
sfhq_symlink = FIUBENCH_DIR / "dataset/SFHQ"

existing = list(sfhq_dir.glob("*.jpg")) if sfhq_dir.exists() else []
if len(existing) >= 400:
    print(f"✅ SFHQ images present: {len(existing)}")
else:
    print(f"⬇  SFHQ missing ({len(existing)}) — re-downloading...")
    from huggingface_hub import hf_hub_download
    sfhq_dir.mkdir(parents=True, exist_ok=True)
    zip_path = hf_hub_download(repo_id="gray311/FIUBench", filename="SFHQ.zip", repo_type="dataset")
    extract_dir = sfhq_dir.parent / "sfhq_extracted"
    extract_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(extract_dir)
    for src in extract_dir.rglob("*.jpg"):
        dst = sfhq_dir / src.name
        if not dst.exists():
            shutil.copy2(src, dst)
    print(f"✅ SFHQ ready: {len(list(sfhq_dir.glob('*.jpg')))} images")

if sfhq_symlink.is_symlink() and not sfhq_symlink.exists():
    sfhq_symlink.unlink()
if not sfhq_symlink.exists():
    sfhq_symlink.parent.mkdir(parents=True, exist_ok=True)
    sfhq_symlink.symlink_to(sfhq_dir)
    print("✅ SFHQ symlink created")
else:
    print("✅ SFHQ symlink OK")

os.chdir(str(FIUBENCH_DIR))

# ── Generation configs to test ────────────────────────────────────────────────
CONFIGS = [
    ("greedy",   False, None),
    ("temp=0.1", True,  0.1),
    ("temp=0.3", True,  0.3),
    ("temp=0.7", True,  0.7),
    ("temp=1.0", True,  1.0),
]

results = {}

for label, do_sample, temperature in CONFIGS:
    save_dir = f"../results/temp_sweep/{label.replace('=','_')}"
    Path(save_dir).mkdir(parents=True, exist_ok=True)

    cmd = [
        'python', 'evaluate_util.py', '--config-name', 'eval',
        f'model_path={MODEL_PATH}',
        'LoRA.r=0',
        f'save_dir={save_dir}',
        'split_list=[retain5]',
        'eval_task=[eval_retain_log]',
        'robust_eval=[[rouge]]',
        'batch_size=1',
        'perturb_batch_size=1',
        'generation.max_new_tokens=50',
        f'generation.do_sample={str(do_sample).lower()}',
        'overwrite=true',
        f'hydra.run.dir=/tmp/hydra_sweep_{label.replace("=","_")}',
    ]
    if temperature is not None:
        cmd.append(f'generation.temperature={temperature}')

    print(f"\n{'─'*50}")
    print(f"Running: {label}")
    print(f"{'─'*50}")

    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
    if proc.returncode != 0:
        print(f"❌ FAILED (exit {proc.returncode})")
        results[label] = None
        continue

    result_file = Path(save_dir) / 'retain5_eval_retain_log.json'
    if not result_file.exists():
        print(f"❌ Result file not found")
        results[label] = None
        continue

    data   = json.load(open(result_file))
    rouge  = data.get('rougeL_recall', {})
    scores = [float(v) for v in rouge.values() if v is not None]
    mean_rouge = np.mean(scores) * 100
    results[label] = mean_rouge
    print(f"✅ ROUGE-L recall: {mean_rouge:.1f}%")

# ── Summary table ─────────────────────────────────────────────────────────────
print(f"\n{'='*50}")
print(f"{'Config':<14} {'ROUGE-L':>10}  {'vs greedy':>10}")
print(f"{'─'*50}")
baseline = results.get("greedy")
for label, score in results.items():
    if score is None:
        print(f"{label:<14} {'FAILED':>10}")
    else:
        delta = f"{score - baseline:+.1f}pp" if baseline is not None and label != "greedy" else "—"
        print(f"{label:<14} {score:>9.1f}%  {delta:>10}")
print(f"{'─'*50}")
print(f"{'Paper target':<14} {'93.3':>10}%")
print(f"{'='*50}")
if baseline is not None:
    valid = {l: s for l, s in results.items() if s is not None}
    best_label = max(valid, key=lambda l: valid[l])
    best_score = valid[best_label]
    if best_score > baseline + 1.0:
        print(f"\n→ Temperature matters: best={best_label} ({best_score:.1f}%), greedy={baseline:.1f}%")
    else:
        print(f"\n→ Temperature has minimal effect — training gap is the dominant factor")


## Stage 1 Decoding Temperature Analysis
*Target Split: `retain5` (400 samples)*

| Configuration | ROUGE-L | vs greedy |
|:---|:---|:---|
| **greedy** | **76.6%** | **—** |
| temp=0.1 | 76.6% | -0.0pp |
| temp=0.3 | 75.9% | -0.7pp |
| temp=0.7 | 73.2% | -3.4pp |
| temp=1.0 | 69.5% | -7.1pp |
| **Paper target** | **93.3%** | |

### So temperature close to do_sample=False or 0.1 is best for high Rouge-L score 

## GPT-Eval Wrapper (GPT-4o-mini)

Same prompt as `FIUBench/gpt_eval.py`. Scores each prediction 0–1 for factual accuracy.

In [ ]:
import os, re, time
from openai import OpenAI

_GPT_PROMPT = """You are an intelligent chatbot designed for evaluating the factual accuracy of generative outputs for question-answer pairs about fictitious entities.
Your task is to compare the predicted answer with the correct answer and determine if they are factually consistent. Here's how you can accomplish the task:
1. Focus on the meaningful match between the predicted answer and the correct answer.
2. Consider synonyms or paraphrases as valid matches.
3. Evaluate the correctness of the prediction compared to the answer.
4. Please do not consider the difference in sentence style between the correct answer and the predicted answer, but only judge whether the predicted answer makes sense based on factual accuracy.
5. If there is something in the predicted answer that is not in the correct answer, then it is considered to be hallucination.

The score should range from 0 to 1. A larger score means a better answer. The score should be a float number with 2 decimal places.
Please first output a single line containing only one value indicating the scores for the predicted answer.
In the subsequent line, please provide some key words of the question and correct answers.
In the subsequent line, please provide a comprehensive explanation of your evaluation.

Question: {question}
Correct Answer: {answer}
Prediction: {prediction}

Outputs (include score, key words, explanation):"""

def _parse_gpt_score(text: str) -> float:
    line = text.strip().split("\n")[0].strip()
    line = re.sub(r"\*\*", "", line)
    if ":" in line:
        line = line.split(":")[-1].strip()
    try:
        return float(line)
    except ValueError:
        nums = re.findall(r"\d+\.\d+", line)
        return float(nums[0]) if nums else float("nan")

def _extract_question(inp: str) -> str:
    """Extract clean question text from evaluate_util.py input string (handles Phi-3 and LLaVA formats)."""
    if "USER:" in inp:
        q = inp[inp.find("USER:"):].replace("USER:", "").replace("<image>", "").strip()
    elif "<|assistant|>" in inp:
        q = inp.split("<|assistant|>")[0]
        for tok in ["<s>", "<image>", "<|user|>", "<|end|>"]:
            q = q.replace(tok, "")
        q = q.strip()
    else:
        q = inp.split("ASSISTANT")[0].replace("<image>", "").strip()
    return q

def gpt_eval_results(result_json: dict, api_key: str, n: int = 50,
                     model: str = "gpt-4o-mini") -> float:
    """Run GPT-Eval on generated_text from an evaluate_util.py result JSON. Returns mean score."""
    if not api_key:
        print("  ⚠️  No OPENAI_API_KEY — GPT-Eval skipped")
        return float("nan")
    gen_texts = result_json.get("generated_text", {})
    if not gen_texts:
        print("  ⚠️  No generated_text in result")
        return float("nan")
    client = OpenAI(api_key=api_key)
    scores = []
    for idx, val in list(gen_texts.items())[:n]:
        inp, gen, gt, cat = val
        q = _extract_question(inp)
        prompt = _GPT_PROMPT.format(question=q, answer=gt, prediction=gen)
        for attempt in range(3):
            try:
                resp = client.chat.completions.create(
                    model=model,
                    messages=[{"role": "user", "content": prompt}],
                    max_tokens=100, temperature=0.0,
                )
                scores.append(_parse_gpt_score(resp.choices[0].message.content))
                break
            except Exception as e:
                if attempt == 2:
                    scores.append(float("nan"))
                else:
                    time.sleep(2 ** attempt)
    valid = [s for s in scores if s == s]
    mean = sum(valid) / len(valid) if valid else float("nan")
    print(f"  GPT-Eval: {len(valid)}/{len(scores)} scored, mean = {mean:.3f}")
    return mean

# Smoke tests
assert abs(_parse_gpt_score("0.85\nkeywords\nexplanation") - 0.85) < 1e-6
assert _extract_question('<s> \nWhat is the blood type?<|end|><|assistant|>') == 'What is the blood type?'
assert _extract_question('USER: <image>\nWhat is the name?ASSISTANT:') == 'What is the name?ASSISTANT:'
print("✅ GPT-Eval wrapper ready")

✅ GPT-Eval wrapper ready


## Table 1 — Pretrained vs Finetuned (Rouge-L + GPT-Eval)

Loads already-saved eval JSON files (no re-inference needed) and runs GPT-Eval on the generated outputs.

In [ ]:
import json, numpy as np
from pathlib import Path

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")  # set this before running
GPT_N = 50  # samples to score per model (full=400, use 50 for cost control)

EVALS = {
    "Pretrained":  Path("/content/FIUBench_Reproducing/results/baseline_eval_retain5/retain5_eval_retain_log.json"),
    "Finetuned":   Path("/content/FIUBench_Reproducing/results/stage1_eval_retain5/retain5_eval_retain_log.json"),
}

results = {}
for label, path in EVALS.items():
    print(f"\n── {label} ──────────────────────────────────")
    if not path.exists():
        print(f"  ❌ Result file not found: {path}")
        results[label] = {"rouge_l": float("nan"), "gpt": float("nan")}
        continue

    data   = json.load(open(path))
    rouge  = data.get("rougeL_recall", {})
    scores = [float(v) for v in rouge.values() if v is not None]
    mean_rouge = float(np.mean(scores)) * 100 if scores else float("nan")
    print(f"  Rouge-L (retain5): {mean_rouge:.1f}%")

    gpt_score = gpt_eval_results(data, api_key=OPENAI_API_KEY, n=GPT_N)

    results[label] = {"rouge_l": mean_rouge, "gpt": gpt_score}

# ── Table 1 ───────────────────────────────────────────────────────────────────
print("\n")
print("Table 1: Model performance before and after Stage 1 fine-tuning")
print(f"{'─'*65}")
print(f"{'Model':<28} {'Pretrained':>16} {'':>4} {'Finetuned':>16}")
print(f"{'':28} {'Rouge-L':>8} {'GPT':>8} {'':>4} {'Rouge-L':>8} {'GPT':>8}")
print(f"{'─'*65}")

pre  = results.get("Pretrained",  {})
fine = results.get("Finetuned",   {})

def fmt_rouge(v):
    return f"{v:.1f}" if v == v else "—"
def fmt_gpt(v):
    return f"{v:.2f}" if v == v else "—"
def arrow(pre_v, fine_v, higher_is_better=True):
    if pre_v != pre_v or fine_v != fine_v:
        return ""
    return "↑" if (fine_v > pre_v) == higher_is_better else "↓"

r_arrow = arrow(pre.get("rouge_l", float("nan")), fine.get("rouge_l", float("nan")))
g_arrow = arrow(pre.get("gpt",     float("nan")), fine.get("gpt",     float("nan")))

print(
    f"{'LLaVA-Phi-Mini-3B':<28}"
    f" {fmt_rouge(pre.get('rouge_l', float('nan'))):>8}"
    f" {fmt_gpt(pre.get('gpt',     float('nan'))):>8}"
    f" {'':>4}"
    f" {fmt_rouge(fine.get('rouge_l', float('nan'))) + r_arrow:>8}"
    f" {fmt_gpt(fine.get('gpt',     float('nan'))) + g_arrow:>8}"
)
print(f"{'─'*65}")
print(f"Paper (LLaVA-Phi-Mini-3B)   {'53.6':>8} {'0.07':>8} {'':>4} {'93.3↑':>8} {'85.8↑':>8}")
print(f"{'─'*65}")


── Pretrained ──────────────────────────────────
  Rouge-L (retain5): 58.6%
  GPT-Eval: 50/50 scored, mean = 0.076

── Finetuned ──────────────────────────────────
  Rouge-L (retain5): 76.5%
  GPT-Eval: 50/50 scored, mean = 0.220


Table 1: Model performance before and after Stage 1 fine-tuning
─────────────────────────────────────────────────────────────────
Model                              Pretrained             Finetuned
                              Rouge-L      GPT       Rouge-L      GPT
─────────────────────────────────────────────────────────────────
LLaVA-Phi-Mini-3B                58.6     0.08         76.5↑    0.22↑
─────────────────────────────────────────────────────────────────
Paper (LLaVA-Phi-Mini-3B)       53.6     0.07         93.3↑    85.8↑
─────────────────────────────────────────────────────────────────


In [ ]:
import subprocess
import os
import json
import numpy as np
from pathlib import Path

PROJECT_ROOT = '/content/FIUBench_Reproducing'
FIUBENCH_DIR = Path(PROJECT_ROOT) / 'FIUBench'
CHECKPOINT_DIR = '/content/stage1_checkpoints'  # or your finetuned checkpoint
VENV_PYTHON = "/content/py310_env/bin/python"

os.chdir(str(FIUBENCH_DIR))

print("Evaluating on FORGET set (S_F)...\n")

# Run evaluation on forget5 split
proc = subprocess.Popen([
    VENV_PYTHON, 'evaluate_util.py',
    '--config-name', 'eval',
    f'model_path={CHECKPOINT_DIR}',
    'LoRA.r=0',
    'save_dir=../results/stage1_eval_forget5',
    'split_list=[forget5]',           # ← FORGET set, not retain
    'eval_task=[eval_forget_log]',    # ← eval_forget_log task
    'robust_eval=[[rouge]]',
    'batch_size=1',
    'perturb_batch_size=1',
    'generation.max_new_tokens=50',
    'overwrite=true',
    'hydra.run.dir=/tmp/hydra_stage1_eval_forget'
], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()

# Parse results
result_path = Path('../results/stage1_eval_forget5/forget5_eval_forget_log.json')
if result_path.exists():
    data = json.load(open(result_path))
    rouge = data.get('rougeL_recall', {})
    scores = [float(v) for v in rouge.values() if v is not None]
    mean_rouge = np.mean(scores) * 100
    
    print(f"\n{'='*60}")
    print(f"FORGET SET (S_F) - Before Unlearning:")
    print(f"  ROUGE-L: {mean_rouge:.1f}%")
    print(f"  (Low score = model forgot info - GOOD)")
    print(f"  (High score = model still knows info - BAD)")
    print(f"{'='*60}")
else:
    print("❌ Evaluation failed")

Evaluating on FORGET set (S_F)...

/content/py310_env/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
[2026-04-19 00:14:37,171] [INFO] [real_accelerator.py:203:get_accelerator] Setting ds_accelerator to cuda (auto detect)
 [WARNING]  async_io requires the dev libaio .so object and headers but these were not found.
 [WARNING]  async_io: please install the libaio-dev package with apt
 [WARNING]  If libaio is already installed (perhaps from source), try setting the CFLAGS and LDFLAGS environment variables to where it can be found.
 [WARNING]  Please specify the CUTLASS repo directory as environment variable $CUTLASS_PATH
 [WARNING]  sparse_attn requires a torch version >= 1.5 and < 2.0 but detected 2.2
 [WARNING]  using untested triton

In [ ]:
from transformers import LlavaForConditionalGeneration
import torch

ckpt = LlavaForConditionalGeneration.from_pretrained(CHECKPOINT_DIR)
base = LlavaForConditionalGeneration.from_pretrained("xtuner/llava-phi-3-mini-hf")

# Count how many layers actually changed
changed, frozen = 0, 0
for (n1, p1), (n2, p2) in zip(ckpt.named_parameters(), base.named_parameters()):
    if torch.equal(p1.cpu(), p2.cpu()):
        frozen += 1
    else:
        changed += 1

print(f"Changed layers: {changed}")
print(f"Frozen layers:  {frozen}")
print(f"Total layers:   {changed + frozen}")

Loading weights:   0%|          | 0/686 [00:00<?, ?it/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/686 [00:00<?, ?it/s]

Changed layers: 250
Frozen layers:  436
Total layers:   686


In [ ]:
from transformers import AutoTokenizer
from utils import get_model_identifiers_from_yaml

tokenizer = AutoTokenizer.from_pretrained("xtuner/llava-phi-3-mini-hf")
model_cfg = get_model_identifiers_from_yaml("llava-phi")

print("system_tag:        ", repr(model_cfg['system_tag']))
print("question_start_tag:", repr(model_cfg['question_start_tag']))
print("answer_tag:        ", repr(model_cfg['answer_tag']))

# Build a sample conversation and check what gets masked
question = "What is the full name of the person in the image?"
answer = "The full name of the person in the image is Jody Vance."

roles = [model_cfg['question_start_tag'], model_cfg['answer_tag']]
conversation = model_cfg['system_tag'] + roles[0] + "<image>\n" + question + roles[1] + answer

print("\nFull conversation:")
print(repr(conversation))

# Tokenize and check labels
text_input = tokenizer(conversation, return_tensors="pt")
from data_module import preprocess_v1
labels = preprocess_v1(tokenizer, text_input['input_ids'], conversation, roles)

# Show what is masked vs what is trained
tokens = tokenizer.convert_ids_to_tokens(text_input['input_ids'][0])
label_ids = labels[0].tolist()

print("\nToken → Label (what model actually trains on):")
for i, (tok, lab) in enumerate(zip(tokens, label_ids)):
    status = "TRAIN" if lab != -100 else "masked"
    print(f"  {i:3d} | {tok:20s} | {status}")

system_tag:         ''
question_start_tag: '<|user|>\n'
answer_tag:         '<|end|>\n<|assistant|>\n'

Full conversation:
'<|user|>\n<image>\nWhat is the full name of the person in the image?<|end|>\n<|assistant|>\nThe full name of the person in the image is Jody Vance.'

Token → Label (what model actually trains on):
    0 | <s>                  | masked
    1 | <|user|>             | masked
    2 | <image>              | masked
    3 | ▁                    | masked
    4 | <0x0A>               | masked
    5 | What                 | masked
    6 | ▁is                  | masked
    7 | ▁the                 | masked
    8 | ▁full                | masked
    9 | ▁name                | masked
   10 | ▁of                  | masked
   11 | ▁the                 | masked
   12 | ▁person              | masked
   13 | ▁in                  | masked
   14 | ▁the                 | masked
   15 | ▁image               | masked
   16 | ?                    | masked
   17 | <|end|>              | ma

In [ ]:
from transformers import LlavaForConditionalGeneration
import torch

ckpt = LlavaForConditionalGeneration.from_pretrained(CHECKPOINT_DIR)
base = LlavaForConditionalGeneration.from_pretrained("xtuner/llava-phi-3-mini-hf")

# Count how many layers actually changed
changed, frozen = 0, 0
for (n1, p1), (n2, p2) in zip(ckpt.named_parameters(), base.named_parameters()):
    if torch.equal(p1.cpu(), p2.cpu()):
        frozen += 1
    else:
        changed += 1

print(f"Changed layers: {changed}")
print(f"Frozen layers:  {frozen}")
print(f"Total layers:   {changed + frozen}")

Loading weights:   0%|          | 0/686 [00:00<?, ?it/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/686 [00:00<?, ?it/s]

Changed layers: 250
Frozen layers:  436
Total layers:   686


In [ ]:
vision_trainable = sum(
    p.numel() for n, p in model.named_parameters() 
    if 'vision' in n and p.requires_grad
)
print(f"Vision tower trainable params: {vision_trainable:,}")
# Must print a large number, NOT 0

Vision tower trainable params: 303,507,456
